<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/dbunet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"]='KGAT_c03d989b55c966d18c971a92b023645b'

In [ ]:
!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:12<00:00, 15.9MB/s]



In [ ]:
!unzip -q busi-dataset.zip -d busi_dataset

In [ ]:
!pip install albumentations scikit-learn -q

import os
import copy
import cv2
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import albumentations as A
from sklearn.model_selection import KFold
from tqdm import tqdm

In [ ]:
BATCH_SIZE = 16
EPOCHS = 50
L0 = 0.0001
PATIENCE = 3
DECAY_FACTOR = 0.2

IMG_SIZE = 256
BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

In [ ]:

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'image_edge': 'image'})

test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
], additional_targets={'image_edge': 'image'})


class DBUSIDataset(Dataset):
    def __init__(self, base_dir, classes, transform=None):
        self.samples = []
        self.transform = transform
        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir): continue
            images = [f for f in os.listdir(cls_dir) if f.endswith(".png") and "_mask" not in f]
            for img_name in images:
                img_path = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = [f for f in os.listdir(cls_dir) if f.startswith(base_name + "_mask") and f.endswith(".png")]
                if not mask_files: continue
                self.samples.append((img_path, [os.path.join(cls_dir, f) for f in mask_files]))

    def __len__(self): return len(self.samples)

    def get_roberts_edge(self, img_gray):
        kernel_x = np.array([[1, 0], [0, -1]], dtype=np.float32)
        kernel_y = np.array([[0, 1], [-1, 0]], dtype=np.float32)
        edge_x = cv2.filter2D(img_gray, cv2.CV_32F, kernel_x)
        edge_y = cv2.filter2D(img_gray, cv2.CV_32F, kernel_y)
        edge = np.sqrt(np.square(edge_x) + np.square(edge_y))
        return cv2.normalize(edge, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    def __getitem__(self, idx):
        img_path, mask_paths = self.samples[idx]

        image_raw = cv2.imread(img_path)
        image_raw = cv2.cvtColor(image_raw, cv2.COLOR_BGR2RGB)

        gray = cv2.cvtColor(image_raw, cv2.COLOR_RGB2GRAY)
        edge_single = self.get_roberts_edge(gray)
        image_edge = np.stack([edge_single]*3, axis=-1)

        combined_mask = np.zeros(image_raw.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            mask = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, (mask > 0).astype(np.uint8))
        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image_raw, image_edge=image_edge, mask=combined_mask)
            image_raw, image_edge, combined_mask = augmented["image"], augmented["image_edge"], augmented["mask"]

        t_raw = torch.from_numpy(image_raw).permute(2, 0, 1).float()
        t_edge = torch.from_numpy(image_edge).permute(2, 0, 1).float()
        t_mask = torch.from_numpy(combined_mask).unsqueeze(0).float()

        return t_raw, t_edge, t_mask


full_dataset = DBUSIDataset(BASE_DIR, classes=CLASSES, transform=None)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class DynamicFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.w_raw = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.5)
        self.w_edge = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.5)
    def forward(self, x_raw, x_edge):
        return (self.w_raw * x_raw) + (self.w_edge * x_edge)

class DBUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.raw_enc1 = DoubleConv(in_channels, 64)
        self.raw_enc2 = DoubleConv(64, 128)
        self.raw_enc3 = DoubleConv(128, 256)
        self.raw_enc4 = DoubleConv(256, 512)

        self.edge_enc1 = DoubleConv(in_channels, 64)
        self.edge_enc2 = DoubleConv(64, 128)
        self.edge_enc3 = DoubleConv(128, 256)
        self.edge_enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)
        self.fuse1 = DynamicFusion(64)
        self.fuse2 = DynamicFusion(128)
        self.fuse3 = DynamicFusion(256)
        self.fuse4 = DynamicFusion(512)

        self.raw_bot = DoubleConv(512, 1024)
        self.edge_bot = DoubleConv(512, 1024)
        self.fuse_bot = DynamicFusion(1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x_raw, x_edge):
        r1, e1 = self.raw_enc1(x_raw), self.edge_enc1(x_edge)
        f1 = self.fuse1(r1, e1)

        r2, e2 = self.raw_enc2(self.pool(r1)), self.edge_enc2(self.pool(e1))
        f2 = self.fuse2(r2, e2)

        r3, e3 = self.raw_enc3(self.pool(r2)), self.edge_enc3(self.pool(e2))
        f3 = self.fuse3(r3, e3)

        r4, e4 = self.raw_enc4(self.pool(r3)), self.edge_enc4(self.pool(e3))
        f4 = self.fuse4(r4, e4)

        rb, eb = self.raw_bot(self.pool(r4)), self.edge_bot(self.pool(e4))
        fb = self.fuse_bot(rb, eb)

        d4 = self.dec4(torch.cat([self.up4(fb), f4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), f3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), f2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), f1], dim=1))

        return self.final_conv(d1)

In [ ]:

class StrictFocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, focal_weight=0.5, smooth=1e-5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
        self.focal_weight, self.dice_weight = focal_weight, 1.0 - focal_weight
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt = probs * targets + (1 - probs) * (1 - targets)
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * bce_loss).mean()

        probs_flat, targets_flat = probs.view(-1), targets.view(-1)
        inter = (probs_flat * targets_flat).sum()
        dice_loss = 1.0 - (2. * inter + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)

        return (self.focal_weight * focal_loss) + (self.dice_weight * dice_loss)

def strict_dice_coef(y_true, logits, smooth=1e-5):
    y_pred = (torch.sigmoid(logits) > 0.5).float().view(-1)
    y_true_f = y_true.view(-1)
    inter = (y_true_f * y_pred).sum()
    return (2. * inter + smooth) / (y_true_f.sum() + y_pred.sum() + smooth)

In [ ]:

kfold = KFold(n_splits=5, shuffle=True, random_state=999)
fold_results = []

print("\n INITIATING 5-FOLD CROSS-VALIDATION")
print(f"Hyperparameters: L0={L0}, Batch={BATCH_SIZE}, Epochs={EPOCHS}")
print("=" * 70)

for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):
    print(f"\n--- STARTING FOLD {fold+1}/5 ---")

    train_sub = Subset(DBUSIDataset(BASE_DIR, CLASSES, transform=train_transform), train_ids)
    test_sub = Subset(DBUSIDataset(BASE_DIR, CLASSES, transform=test_transform), test_ids)

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = DBUNet(in_channels=3, out_channels=1).to(device)
    criterion = StrictFocalDiceLoss(alpha=0.25, gamma=2.0, focal_weight=0.5)

    optimizer = torch.optim.Adam(model.parameters(), lr=L0)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=DECAY_FACTOR, patience=PATIENCE)
    scaler = torch.amp.GradScaler('cuda')

    best_test_dice_for_fold = 0.0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0

        for img_raw, img_edge, masks in tqdm(train_loader, desc=f"Fold {fold+1} | Epoch {epoch+1}/{EPOCHS}", leave=False):
            img_raw, img_edge, masks = img_raw.to(device, non_blocking=True), img_edge.to(device, non_blocking=True), masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                logits = model(img_raw, img_edge)
                loss = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()


        model.eval()
        test_dice = 0
        with torch.no_grad():
            for img_raw, img_edge, masks in test_loader:
                img_raw, img_edge, masks = img_raw.to(device, non_blocking=True), img_edge.to(device, non_blocking=True), masks.to(device, non_blocking=True)
                with torch.amp.autocast('cuda'):
                    logits = model(img_raw, img_edge)
                test_dice += strict_dice_coef(masks, logits).item()

        avg_test_dice = test_dice / len(test_loader)
        avg_train_loss = train_loss / len(train_loader)


        scheduler.step(avg_test_dice)

        if avg_test_dice > best_test_dice_for_fold:
            best_test_dice_for_fold = avg_test_dice

    print(f" Fold {fold+1} Completed | Best Test Dice: {best_test_dice_for_fold:.4f}")
    fold_results.append(best_test_dice_for_fold)


print("\n" + "=" * 70)
print(f" FINAL 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 70)
for idx, res in enumerate(fold_results):
    print(f"Fold {idx+1}: {res:.4f}")

mean_dice = np.mean(fold_results)
std_dice = np.std(fold_results)
print(f"\n FINAL MEAN DICE SCORE: {mean_dice:.4f} \u00b1 {std_dice:.4f}")



 INITIATING 5-FOLD CROSS-VALIDATION
Hyperparameters: L0=0.0001, Batch=16, Epochs=50

--- STARTING FOLD 1/5 ---


 Fold 1 Completed | Best Test Dice: 0.7997

--- STARTING FOLD 2/5 ---


 Fold 2 Completed | Best Test Dice: 0.7291

--- STARTING FOLD 3/5 ---


 Fold 3 Completed | Best Test Dice: 0.7358

--- STARTING FOLD 4/5 ---


 Fold 4 Completed | Best Test Dice: 0.8108

--- STARTING FOLD 5/5 ---


 Fold 5 Completed | Best Test Dice: 0.6919

 FINAL 5-FOLD CROSS-VALIDATION RESULTS
Fold 1: 0.7997
Fold 2: 0.7291
Fold 3: 0.7358
Fold 4: 0.8108
Fold 5: 0.6919

 FINAL MEAN DICE SCORE: 0.7535 ± 0.0450
